In [2]:
from pathlib import Path
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from loaders import TiNADataLoader, standardize_train_test
from autoencoders import RecurrentAutoencoder, ModelConfig as RAEConfig
from vae import (
    VariationalRecurrentAutoencoder,
    ModelConfig as VAEConfig,
    train_vrae,
)

In [13]:
DATA_DIR = Path('./tina/data')
MODEL_DIR = Path('./models/tina')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = 30
STRIDE = 5
RESAMPLE_SECONDS = 10
MAX_FILES = 500
MAX_WINDOWS = 100_000 
TRAIN_SIZE = 0.7
SEED = 42
BATCH_SIZE = 64
EPOCHS_RAE = 30
EPOCHS_VAE = 30
RUN_RAE = True
RUN_VAE = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda:3


In [14]:
loader = TiNADataLoader(
    DATA_DIR,
    label_source='alarms',
    resample_seconds=RESAMPLE_SECONDS,
    max_files=MAX_FILES,
    max_samples=MAX_WINDOWS,
    stride=STRIDE,
    extract_stats=False,
)


X_parts, y_parts, event_parts, id_parts = [], [], [], []
total = 0
for sequence in loader.iter_sequences():
    if len(sequence.X) < WINDOW:
        continue
    seq_X = np.lib.stride_tricks.sliding_window_view(
        sequence.X, window_shape=WINDOW, axis=0
    ).transpose(0, 2, 1)[::STRIDE]
    seq_y_windows = np.lib.stride_tricks.sliding_window_view(
        np.asarray(sequence.y, dtype=bool), window_shape=WINDOW
    )[::STRIDE]
    seq_y_endpoint = seq_y_windows[:, -1]
    seq_has_alarm = seq_y_windows.any(axis=1)
    remaining = MAX_WINDOWS - total
    if remaining <= 0:
        break
    take = min(len(seq_X), remaining)
    if take:
        X_parts.append(np.asarray(seq_X[:take], dtype=np.float32))
        y_parts.append(seq_y_endpoint[:take])
        event_parts.append(seq_has_alarm[:take])
        id_parts.append(np.full(take, sequence.sequence_id, dtype=object))
        total += take

if not X_parts:
    raise ValueError('Не найдено ни одного полного окна')
X = np.concatenate(X_parts)
y_endpoint = np.concatenate(y_parts)
window_has_alarm = np.concatenate(event_parts)
sequence_ids = np.concatenate(id_parts)
print('X:', X.shape, X.dtype)
print('endpoint alarm share:', y_endpoint.mean())
print('windows containing alarm:', window_has_alarm.mean())
print('sequences:', len(np.unique(sequence_ids)))

X: (100000, 30, 103) float32
endpoint alarm share: 0.15715
windows containing alarm: 0.17543
sequences: 488


In [15]:
ordered_groups = list(dict.fromkeys(sequence_ids.tolist()))
if len(ordered_groups) < 2:
    raise ValueError('Нужно хотя бы две последовательности; увеличь MAX_FILES')
cut = min(max(1, int(len(ordered_groups) * TRAIN_SIZE)), len(ordered_groups) - 1)
train_groups = set(ordered_groups[:cut])
train_mask = np.fromiter((g in train_groups for g in sequence_ids), dtype=bool)
val_mask = ~train_mask

# AE не должен учиться восстанавливать alarm; проверка тоже normal-only.
train_idx = np.flatnonzero(train_mask & ~window_has_alarm)
val_idx = np.flatnonzero(val_mask & ~window_has_alarm)
if len(train_idx) == 0 or len(val_idx) == 0:
    raise ValueError('После normal-only фильтра train или validation пуст')

X_train, X_val, scaler = standardize_train_test(X[train_idx], X[val_idx])
del X
print('normal train:', X_train.shape)
print('normal validation:', X_val.shape)
print('scaled train mean/std:', float(X_train.mean()), float(X_train.std()))

normal train: (50039, 30, 103)
normal validation: (32418, 30, 103)
scaled train mean/std: 3.233723733231386e-09 1.0


In [16]:
# DataLoader над Tensor возвращает сам Tensor, что совместимо с текущими train-функциями.
train_loader = DataLoader(
    torch.from_numpy(X_train), batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=(device.type == 'cuda'),
)
val_loader = DataLoader(
    torch.from_numpy(X_val), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=(device.type == 'cuda'),
)
INPUT_DIM = X_train.shape[-1]
print('input_dim:', INPUT_DIM, 'batches:', len(train_loader), len(val_loader))

input_dim: 103 batches: 782 507


In [10]:
def train_rae(model, train_loader, val_loader, config, saved_path, epochs=30, lr=1e-3, patience=7):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    history = {'train_loss': [], 'val_loss': []}
    best_val = float('inf')
    no_improve = 0

    for epoch in range(1, epochs + 1):
        started = time.time()
        model.train()
        train_sum = 0.0
        for batch in train_loader:
            batch = batch.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            reconstructed, _ = model(batch)
            loss = F.mse_loss(reconstructed, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_sum += loss.item() * len(batch)

        model.eval()
        val_sum = 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device, non_blocking=True)
                reconstructed, _ = model(batch)
                val_sum += F.mse_loss(reconstructed, batch).item() * len(batch)

        train_loss = train_sum / len(train_loader.dataset)
        val_loss = val_sum / len(val_loader.dataset)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f'RAE {epoch:02d}/{epochs}: train={train_loss:.6f} val={val_loss:.6f} time={time.time()-started:.1f}s')

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            no_improve = 0
            model.to('cpu')
            model.save_pretrained(str(saved_path), config)
            model.to(device)
        else:
            no_improve += 1
            if no_improve >= patience:
                print('RAE early stopping')
                break
    return history

In [19]:
rae_history = None
rae_path = MODEL_DIR / 'tina_rae_w30_l2.pth'
if RUN_RAE:
    rae_config = RAEConfig(
        input_dim=INPUT_DIM, window_size=WINDOW, hidden_dim=64, latent_dim=32,
        num_layers=2, rnn_type='GRU', dropout=0.1,
        description='TiNA, 10 s mean, normal-only, window 30',
        num_layers_linear=1,
    )
    rae = RecurrentAutoencoder(config=rae_config)
    rae_history = train_rae(
        rae, train_loader, val_loader, rae_config, rae_path,
        epochs=EPOCHS_RAE, patience=7,
    )
    print('saved:', rae_path)

RAE 01/30: train=0.260479 val=1.581962 time=6.1s
RAE 02/30: train=0.176985 val=1.508661 time=4.5s
RAE 03/30: train=0.159960 val=1.578826 time=4.9s
RAE 04/30: train=0.152247 val=1.577785 time=4.7s
RAE 05/30: train=0.147484 val=1.552668 time=4.8s
RAE 06/30: train=0.143383 val=1.512727 time=4.7s
RAE 07/30: train=0.139591 val=1.487634 time=4.5s
RAE 08/30: train=0.137016 val=1.456619 time=4.5s
RAE 09/30: train=0.134690 val=1.451853 time=4.8s
RAE 10/30: train=0.133294 val=1.428040 time=4.8s
RAE 11/30: train=0.131855 val=1.420455 time=4.8s
RAE 12/30: train=0.130644 val=1.407292 time=4.5s
RAE 13/30: train=0.130378 val=1.409068 time=4.5s
RAE 14/30: train=0.128366 val=1.402097 time=4.7s
RAE 15/30: train=0.127613 val=1.401313 time=4.6s
RAE 16/30: train=0.127021 val=1.426191 time=4.5s
RAE 17/30: train=0.126075 val=1.413312 time=4.7s
RAE 18/30: train=0.125437 val=1.434097 time=4.9s
RAE 19/30: train=0.124416 val=1.443764 time=4.8s
RAE 20/30: train=0.123970 val=1.446379 time=4.7s
RAE 21/30: train=0.1

In [ ]:
rae_history = None
rae_path = MODEL_DIR / 'tina_rae_w30_64_16.pth'
if RUN_RAE:
    rae_config = RAEConfig(
        input_dim=INPUT_DIM, window_size=WINDOW, hidden_dim=128, latent_dim=16,
        num_layers=1, rnn_type='GRU', dropout=0.1,
        description='TiNA, 10 s mean, normal-only, window 30',
        num_layers_linear=1,
    )
    rae = RecurrentAutoencoder(config=rae_config)
    rae_history = train_rae(
        rae, train_loader, val_loader, rae_config, rae_path,
        epochs=EPOCHS_RAE, patience=7,
    )
    print('saved:', rae_path)

RAE 01/30: train=0.240516 val=1.521257 time=3.8s
RAE 02/30: train=0.158697 val=1.515312 time=3.9s
RAE 03/30: train=0.148902 val=1.541158 time=4.2s
RAE 04/30: train=0.145163 val=1.560570 time=4.1s
RAE 05/30: train=0.142787 val=1.558472 time=3.9s
RAE 06/30: train=0.141250 val=1.553968 time=4.1s
RAE 07/30: train=0.139985 val=1.552829 time=3.8s
RAE 08/30: train=0.138869 val=1.554600 time=3.9s
RAE 09/30: train=0.137976 val=1.550020 time=4.2s
RAE early stopping
saved: models/tina/tina_rae_w30_64_16.pth


In [21]:
vae_train_metrics = vae_val_metrics = None
vae_path = MODEL_DIR / 'tina_vae_w30_64.pth'
if RUN_VAE:
    vae_config = VAEConfig(
        input_dim=INPUT_DIM, window_size=WINDOW, hidden_dim=64, latent_dim=32,
        num_layers=1, rnn_type='GRU', dropout=0.1,
        description='TiNA, 10 s mean, normal-only, window 30',
    )
    vae_model = VariationalRecurrentAutoencoder(config=vae_config)
    vae_train_metrics, vae_val_metrics = train_vrae(
        model=vae_model, config=vae_config,
        train_loader=train_loader, test_loader=val_loader,
        epochs=EPOCHS_VAE, lr=1e-3, beta=0.01,
        beta_warmup=True, warmup_steps=max(1, 5 * len(train_loader)),
        device=device, saved_path=str(vae_path), patience=7,
    )
    print('saved:', vae_path)

Начинаем обучение VRAE на 30 эпох
Device: cuda:3, Learning rate: 0.001, Beta: 0.01, Beta warmup: True
--------------------------------------------------------------------------------
Epoch   1/30 [5.8s]
  Train - Loss: 0.296689, Recon: 0.267273, KL: 32.475652, Beta: 0.0020
  Test  - Loss: 1.630674, Recon: 1.580219, KL: 25.259637
Model saved  → models/tina/tina_vae_w30_64.pth
Config saved → models/tina/tina_vae_w30_64_config.json
  Модель сохранена в models/tina/tina_vae_w30_64.pth (loss: 1.630674)
  Ratio test/train: 5.496
--------------------------------------------------------------------------------
Epoch   2/30 [5.4s]
  Train - Loss: 0.242785, Recon: 0.194091, KL: 16.678988, Beta: 0.0040
  Test  - Loss: 1.705471, Recon: 1.636606, KL: 17.227324
  Нет улучшения 1/7 
  Ratio test/train: 7.025
--------------------------------------------------------------------------------
Epoch   3/30 [5.7s]
  Train - Loss: 0.252557, Recon: 0.193127, KL: 11.988219, Beta: 0.0060
  Test  - Loss: 1.83634